# **PHASE 8 — GRADIO DEMO**

**Cell 8.1: Xây dựng app demo**

In [3]:
import gradio as gr
import torch
import torch.nn as nn
import timm
import os
import numpy as np
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

BASE = 'D:/DoAnTriTueNhanTao/skin_cancer'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASSES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
MALIGNANT = ['MEL', 'BCC', 'SCC', 'AK']

CLASS_DESC = {
    'MEL': 'Melanoma — U hắc tố (nguy hiểm)',
    'NV':  'Nevus — Nốt ruồi lành tính',
    'BCC': 'Basal Cell Carcinoma — Ung thư tế bào đáy',
    'AK':  'Actinic Keratosis — Dày sừng ánh sáng',
    'BKL': 'Benign Keratosis — Dày sừng lành tính',
    'DF':  'Dermatofibroma — U xơ da',
    'VASC': 'Vascular Lesion — Tổn thương mạch máu',
    'SCC': 'Squamous Cell Carcinoma — Ung thư tế bào vảy',
}

class SkinCancerEfficientNet(nn.Module):
    def __init__(self, backbone='efficientnet_b0', num_classes=2, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=False, num_classes=0)
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 256), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

# Load cả 2 model
model_bin = SkinCancerEfficientNet(num_classes=2, dropout=0.3).to(DEVICE)
model_bin.load_state_dict(torch.load(f'{BASE}/checkpoints/best_binary.pth', map_location=DEVICE))
model_bin.eval()

model_mc = SkinCancerEfficientNet(num_classes=8, dropout=0.4).to(DEVICE)
model_mc.load_state_dict(torch.load(f'{BASE}/checkpoints/best_multiclass.pth', map_location=DEVICE))
model_mc.eval()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict_and_explain(pil_img):
    if pil_img is None:
        return None, None, "Chưa có ảnh", None

    pil_img = pil_img.convert('RGB')
    inp = preprocess(pil_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits_bin = model_bin(inp)
        p_malig = torch.softmax(logits_bin, dim=1)[0, 1].item()

        logits_mc = model_mc(inp)
        probs_mc = torch.softmax(logits_mc, dim=1)[0].cpu().numpy()

    pred_bin_name = "⚠️ UNG THƯ" if p_malig > 0.5 else "✅ LÀNH TÍNH"

    # Text kết quả
    lines = [f"### Kết quả sàng lọc nhị phân",
             f"- Xác suất ung thư: **{p_malig:.2%}**",
             f"- Phân loại: **{pred_bin_name}**",
             "",
             "### Xác suất 8 lớp"]
    for cls, p in zip(CLASSES, probs_mc):
        mark = "🔴" if cls in MALIGNANT else "🟢"
        lines.append(f"- {mark} **{cls}** ({CLASS_DESC[cls]}): {p:.2%}")
    summary = "\n".join(lines)

    # Grad-CAM
    rgb = np.array(pil_img.resize((224, 224))) / 255.0
    with GradCAM(model=model_mc, target_layers=[model_mc.backbone.conv_head]) as cam:
        gray = cam(input_tensor=inp, targets=[ClassifierOutputTarget(int(probs_mc.argmax()))])[0]
    cam_vis = show_cam_on_image(rgb.astype(np.float32), gray, use_rgb=True)

    # Bar chart probs
    fig, ax = plt.subplots(figsize=(6, 3))
    colors = ['#d62728' if c in MALIGNANT else '#2ca02c' for c in CLASSES]
    ax.barh(CLASSES, probs_mc, color=colors)
    ax.set_xlim(0, 1); ax.set_xlabel('Xác suất')
    ax.set_title('Xác suất 8 lớp')
    plt.tight_layout()
    plt.close(fig)

    return cam_vis, fig, summary, float(p_malig)


with gr.Blocks(title="Skin Cancer Classification — EfficientNet-B0") as demo:
    gr.Markdown("# 🔬 Phân loại ung thư da từ ảnh Dermoscopy\n"
                "Mô hình **EfficientNet-B0** — ISIC 2019 (8 lớp, 25,331 ảnh)")

    with gr.Row():
        with gr.Column():
            img_in = gr.Image(type='pil', label='Ảnh dermoscopy')
            btn = gr.Button("Phân tích", variant='primary')
        with gr.Column():
            cam_out = gr.Image(label='Grad-CAM (vùng mô hình chú ý)')
            chart_out = gr.Plot(label='Xác suất 8 lớp')

    text_out = gr.Markdown()
    prob_out = gr.Number(label='Xác suất ung thư', visible=False)

    btn.click(fn=predict_and_explain,
              inputs=img_in,
              outputs=[cam_out, chart_out, text_out, prob_out])

    gr.Examples(
        examples=[
            [f'{BASE}/test_examples/MEL_1.jpg'],
            [f'{BASE}/test_examples/BCC_1.jpg'],
            [f'{BASE}/test_examples/NV_1.jpg'],
        ] if os.path.exists(f'{BASE}/test_examples') else None,
        inputs=img_in,
        label='Ví dụ'
    )

demo.launch(share=True, debug=False)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


**Cell 8.2: Lưu 1 vài ảnh test để làm ví dụ**

In [5]:
import shutil
import pandas as pd
os.makedirs(f'{BASE}/test_examples', exist_ok=True)

mc_pred = pd.read_csv(f'{BASE}/results/multiclass_test_predictions.csv')

for cls in ['MEL', 'BCC', 'NV', 'BKL']:
    sub = mc_pred[(mc_pred['label_name']==cls) & (mc_pred['pred_label_name']==cls)]
    if len(sub):
        row = sub.iloc[0]
        dst = f'{BASE}/test_examples/{cls}_1.jpg'
        shutil.copy(row['img_path'], dst)
        print(f"✓ {dst}")

✓ D:/DoAnTriTueNhanTao/skin_cancer/test_examples/MEL_1.jpg
✓ D:/DoAnTriTueNhanTao/skin_cancer/test_examples/BCC_1.jpg
✓ D:/DoAnTriTueNhanTao/skin_cancer/test_examples/NV_1.jpg
✓ D:/DoAnTriTueNhanTao/skin_cancer/test_examples/BKL_1.jpg
